# Olist Order-Level Reporting

## Goal

Build a monthly delivered-order report safely.

## Core rule

Before every merge or join, identify:

1. What one row represents in the left table.
2. What one row represents in the right table.
3. How many times the join key can appear in each table.
4. What one row represents after the join.
5. Which metrics are safe afterward.

## Table grain

orders:
one row = one order

order_items:
one row = one item inside an order

order_value:
one row = one order after aggregating order_items

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    """
    Walk upward from the current working directory until we find
    the project folders that identify this repository.
    """
    for candidate in (start_path, *start_path.parents):
        has_data_folder = (candidate / "data").exists()
        has_notebooks_folder = (candidate / "notebooks").exists()

        if has_data_folder and has_notebooks_folder:
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure the notebook is opened inside market-intelligence-pipeline."
    )


project_root = find_project_root(Path.cwd().resolve())

raw_olist_dir = project_root / "data" / "raw" / "olist"

print(f"Project root: {project_root}")
print(f"Raw Olist data: {raw_olist_dir}")

def get_raw_file(file_name: str) -> Path:
    """
    Find a file anywhere inside data/raw/olist/.
    """
    matches = list(raw_olist_dir.rglob(file_name))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {file_name} inside {raw_olist_dir}"
        )

    return matches[0]

orders = pd.read_csv(
    get_raw_file("olist_orders_dataset.csv"),
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

order_items = pd.read_csv(
    get_raw_file("olist_order_items_dataset.csv"),
    parse_dates=[
        "shipping_limit_date",
    ],
)


print("orders shape:", orders.shape)
print("order_items shape:", order_items.shape)

orders.head()

Project root: C:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline
Raw Olist data: C:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline\data\raw\olist
orders shape: (99441, 8)
order_items shape: (112650, 7)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [2]:
def run_sql(query: str) -> pd.DataFrame:
    return duckdb.sql(query).df()

In [3]:
# Add gros item value to order items and create order value table
order_items = order_items.copy()

order_items["gross_item_value"] = (
    order_items["price"]
    + order_items["freight_value"]
)

order_value = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        gross_order_value=("gross_item_value", "sum"),
        item_count=("order_item_id", "count"),
        average_item_value=("gross_item_value", "mean"),
    )
)

In [5]:
print("orders order_id unique:") # one row per order
print(orders["order_id"].is_unique)

print("\norder_items order_id unique:") # many rows per order possibly
print(order_items["order_id"].is_unique)

print("\norder_value order_id unique:") # one row per order
print(order_value["order_id"].is_unique)

orders order_id unique:
True

order_items order_id unique:
False

order_value order_id unique:
True


In [ ]:
# Create orders_with_value
# 1. Select relevant order columns
order_report = orders[['order_id','order_status','order_purchase_timestamp']].copy()
# 2. Create purchase_month column formatted YYYY-MM
order_report['purchase_month'] = order_report['order_purchase_timestamp'].dt.to_period('M').astype(str)
# merge order value onto orders
orders_with_value = order_report.merge(
    order_value,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator='item_match'
)
# validate if final df has one row per order
print("Rows in orders:", len(orders))
print("Rows after merge:", len(orders_with_value))
print("Unique order IDs:", orders_with_value["order_id"].nunique())
print("Order IDs remain unique:", orders_with_value["order_id"].is_unique)

orders_with_value["item_match"].value_counts() # if left only, then those rows don't have a matching item row

Rows in orders: 99441
Rows after merge: 99441
Unique order IDs: 99441
Order IDs remain unique: True


item_match
both          98666
left_only       775
right_only        0
Name: count, dtype: int64

In [10]:
orders_with_value.head()

,order_id,order_status,order_purchase_timestamp,purchase_month,gross_order_value,item_count,average_item_value,item_match
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,2017-10,38.71,1.0,38.71,both
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-07,141.46,1.0,141.46,both
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,2018-08,179.12,1.0,179.12,both
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,2017-11,72.20,1.0,72.20,both
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2018-02,28.62,1.0,28.62,both


In [11]:
# get orders with missing items
missing_item_orders = orders_with_value.loc[
    orders_with_value['item_match'].eq('left_only'),
    ['order_id', 'order_status', 'purchase_month']
]

missing_item_orders.head()

,order_id,order_status,purchase_month
266,8e24261a7e58791d10cb1bf9da94df5c,unavailable,2017-11
586,c272bcd21c287498b4883c7512019702,unavailable,2018-01
687,37553832a3a89c9b2db59701c357ca67,unavailable,2017-08
737,d57e15fb07fd180f06ab3926b39edcd2,unavailable,2018-01
1130,00b1cb0320190ca0daa2c88b35206009,canceled,2018-08


In [12]:
# check which order status are missing item data
missing_item_orders.groupby('order_status').agg(
    orders_without_item_data=('order_id','nunique'),
).sort_values('orders_without_item_data', ascending=False)

,orders_without_item_data
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


In [13]:
# filter delivered orders
delivered_orders = orders_with_value[orders_with_value['order_status'] == 'delivered'].copy()
# create a column for missing item data
delivered_orders['missing_item_data'] = delivered_orders['item_match'].eq('left_only').astype(int)
# create monthly summary table
monthly_delivered = delivered_orders.groupby('purchase_month', as_index=False).agg(
    delivered_orders = ('order_id','nunique'),
    total_gross_order_value = ('gross_order_value','sum'),
    average_gross_order_value = ('gross_order_value','mean'),
    average_item_count = ('item_count','mean'),
    delivered_orders_missing_items = ('missing_item_data','sum'),
).sort_values('purchase_month')

monthly_delivered.head()

,purchase_month,delivered_orders,total_gross_order_value,average_gross_order_value,average_item_count,delivered_orders_missing_items
0,2016-09,1,143.46,143.460000,3.000000,0
1,2016-10,265,46490.66,175.436453,1.181132,0
2,2016-12,1,19.62,19.620000,1.000000,0
3,2017-01,750,127482.37,169.976493,1.217333,0
4,2017-02,1653,271239.32,164.089123,1.124017,0
